# Lab 4: MLflow Experiment Tracking and Artifact Management

**Course:** ML Operations  
**Objective:** Set up an MLflow tracking server, log parameters, per-epoch metrics, and artifacts, and manage multiple experiment runs for reproducible ML experimentation.

## 4.1 Setup and Imports

In [1]:
import os
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(repo_root)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

print(f'Working directory: {Path.cwd()}')

Working directory: /home/d1pg1/ml_engineering_labs


In [2]:
import yaml
import mlflow
import torch
import torch.nn as nn

from src.data.dataset import process_data, create_data_loader, train_transform, eval_transform
from src.models.cnn import CifarCNN
from src.training.mlflow_trainer import train_with_mlflow
from src.training.evaluate import test_model

print(f'MLflow version: {mlflow.__version__}')
print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

/home/d1pg1/ml_engineering_labs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow version: 3.12.0
PyTorch version: 2.12.0+cu130
Device: cpu


## 4.2 Part 1: MLflow Tracking Server Setup

### 4.2.1 Start a local MLflow server

Run the command below in a terminal to launch the MLflow UI at `http://localhost:5000`:

```bash
mlflow server \
  --backend-store-uri outputs/mlruns \
  --default-artifact-root outputs/mlruns \
  --host 0.0.0.0 \
  --port 5000
```

For the notebook we use the file-based URI directly — no running server is needed to record runs.

In [3]:
TRACKING_URI = 'outputs/mlruns'
EXPERIMENT_NAME = 'cifar10_mlflow'

Path(TRACKING_URI).mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f'Tracking URI : {mlflow.get_tracking_uri()}')
print(f'Experiment   : {experiment.name}')
print(f'Experiment ID: {experiment.experiment_id}')

/home/d1pg1/ml_engineering_labs/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/14 12:18:32 INFO mlflow.tracking.fluent: Experiment with name 'cifar10_mlflow' does not exist. Creating a new experiment.


Tracking URI : outputs/mlruns
Experiment   : cifar10_mlflow
Experiment ID: 425699227787012270


## 4.3 Part 2: Logging Parameters, Configs, and Metrics

### 4.3.1 Load configuration

In [4]:
with open('configs/base.yaml') as f:
    base_cfg = yaml.safe_load(f)

with open('configs/lab04.yaml') as f:
    lab04_cfg = yaml.safe_load(f)

main_cfg = lab04_cfg['main_run']
print('Base config:', base_cfg)
print('Lab04 main run config:', main_cfg)

Base config: {'data': {'test_size': 0.2, 'val_size': 0.2, 'random_state': 42, 'cifar_url': 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', 'data_dir': 'data', 'n_classes': 10}, 'training': {'lr': 0.001, 'batch_size': 128, 'num_epochs': 20, 'num_workers': 2}, 'output': {'save_path': 'outputs/best_model.pth'}}
Lab04 main run config: {'num_epochs': 10, 'lr': 0.001, 'batch_size': 128, 'run_name': 'Experiment 1 - Stage 2: Model Training'}


### 4.3.2 Prepare datasets (reuse existing extracted CIFAR-10 images)

In [5]:
LABELS_PATH = 'data/cifar-10-batches-py'
IMAGES_DIR  = 'data/images'

train_df, val_df, test_df = process_data(
    images_dir=IMAGES_DIR,
    labels_path=LABELS_PATH,
    config=base_cfg['data'],
)

loader_cfg_train = {
    'batch_size': main_cfg['batch_size'],
    'num_workers': base_cfg['training']['num_workers'],
    'transform': train_transform,
}
loader_cfg_eval = {
    'batch_size': main_cfg['batch_size'],
    'num_workers': base_cfg['training']['num_workers'],
    'transform': eval_transform,
}

train_loader = create_data_loader(IMAGES_DIR, train_df, loader_cfg_train)
val_loader   = create_data_loader(IMAGES_DIR, val_df,   loader_cfg_eval)
test_loader  = create_data_loader(IMAGES_DIR, test_df,  loader_cfg_eval)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

2026-05-14 12:18:32,997 INFO Loading batch: data_batch_1


/home/d1pg1/ml_engineering_labs/src/data/dataset.py:90: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  batch = pickle.load(f, encoding="bytes")


2026-05-14 12:18:33,290 INFO Loading batch: data_batch_2


2026-05-14 12:18:33,554 INFO Loading batch: data_batch_3


2026-05-14 12:18:33,778 INFO Loading batch: data_batch_4


2026-05-14 12:18:33,990 INFO Loading batch: data_batch_5


2026-05-14 12:18:34,240 INFO Loading batch: test_batch


2026-05-14 12:18:34,505 INFO Loaded 60000 images from CIFAR-10 batches.


2026-05-14 12:18:34,557 INFO Prepared 3 data splits — train: 38400, val: 9600, test: 12000


Train batches: 300 | Val batches: 75 | Test batches: 94


### 4.3.3 Main training run — log params and per-epoch metrics

In [6]:
MAIN_SAVE_PATH = Path('outputs/lab04_main.pth')

model = CifarCNN(n_classes=base_cfg['data']['n_classes'])
optimizer = torch.optim.Adam(model.parameters(), lr=main_cfg['lr'])
loss_fn = nn.CrossEntropyLoss()

params_to_log = {
    'lr': main_cfg['lr'],
    'batch_size': main_cfg['batch_size'],
    'num_epochs': main_cfg['num_epochs'],
    'optimizer': 'Adam',
    'test_size': base_cfg['data']['test_size'],
    'val_size': base_cfg['data']['val_size'],
    'random_state': base_cfg['data']['random_state'],
    'n_classes': base_cfg['data']['n_classes'],
    'model': 'CifarCNN',
}

with mlflow.start_run(run_name=main_cfg['run_name']) as run:
    main_run_id = run.info.run_id
    print(f'Run ID: {main_run_id}')

    mlflow.log_params(params_to_log)
    mlflow.log_artifact('configs/base.yaml')
    mlflow.log_artifact('configs/lab04.yaml')

    best_path = train_with_mlflow(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_function=loss_fn,
        optimizer=optimizer,
        num_epochs=main_cfg['num_epochs'],
        device=device,
        save_path=MAIN_SAVE_PATH,
    )

    test_metrics = test_model(model, test_loader, loss_fn, device)
    mlflow.log_metrics({
        'test_loss':      test_metrics['loss'],
        'test_accuracy':  test_metrics['accuracy'],
        'test_precision': test_metrics['precision'],
        'test_recall':    test_metrics['recall'],
        'test_f1':        test_metrics['f1'],
    })

    print('\n=== Main Run Results ===')
    for k, v in test_metrics.items():
        print(f'  {k}: {v:.4f}')

Run ID: d0ac60542e324ed4b4aca677361d5b53


2026-05-14 12:20:03,395 INFO Epoch 1/10, Train Loss: 1.7164


2026-05-14 12:20:09,943 INFO Epoch 1/10, Val Loss: 1.3661


2026-05-14 12:20:09,971 INFO Best model saved with val loss: 1.3661


2026-05-14 12:21:39,943 INFO Epoch 2/10, Train Loss: 1.2906


...

2026-05-14 12:36:40,076 INFO Epoch 10/10, Val Loss: 0.8182


2026-05-14 12:36:40,112 INFO Best model saved with val loss: 0.8182


2026-05-14 12:36:40,113 INFO Training complete.


2026-05-14 12:36:51,186 INFO Test Results — Loss: 0.8205 | Accuracy: 0.7111 | Precision: 0.7166 | Recall: 0.7111 | F1: 0.7049



=== Main Run Results ===
  loss: 0.8205
  accuracy: 0.7111
  precision: 0.7166
  recall: 0.7111
  f1: 0.7049


## 4.4 Part 3: Logging Output Artifacts

Log the best model checkpoint as an MLflow artifact and demonstrate loading it back for inference.

In [7]:
# Log the checkpoint into the existing main run
with mlflow.start_run(run_id=main_run_id):
    mlflow.log_artifact(str(MAIN_SAVE_PATH), artifact_path='model')
    print(f'Artifact logged: {MAIN_SAVE_PATH}')

Artifact logged: outputs/lab04_main.pth


In [8]:
# Demonstrate artifact retrieval: download and run inference
artifact_uri = f'runs:/{main_run_id}/model/lab04_main.pth'
local_path = mlflow.artifacts.download_artifacts(artifact_uri)
print(f'Downloaded to: {local_path}')

loaded_model = CifarCNN(n_classes=base_cfg['data']['n_classes'])
loaded_model.load_state_dict(torch.load(local_path, map_location=device))
loaded_model.eval()

# Run one test batch and show predictions vs ground truth
CIFAR10_CLASSES = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]

sample_inputs, sample_labels = next(iter(test_loader))
sample_inputs = sample_inputs[:8].to(device)
with torch.no_grad():
    preds = torch.argmax(loaded_model(sample_inputs), dim=1).cpu()

print('\nArtifact inference demo (first 8 samples):')
print(f'{"Predicted":<14} {"Ground truth"}')
for p, t in zip(preds.tolist(), sample_labels[:8].tolist()):
    match = '✓' if p == t else '✗'
    print(f'  {CIFAR10_CLASSES[p]:<12} {CIFAR10_CLASSES[t]} {match}')

Downloaded to: /tmp/tmp2mkm74g_/lab04_main.pth



Artifact inference demo (first 8 samples):
Predicted      Ground truth
  ship         ship ✓
  truck        truck ✓
  automobile   cat ✗
  automobile   automobile ✓
  horse        deer ✗
  horse        horse ✓
  horse        horse ✓
  dog          cat ✗


## 4.5 Part 4: Experimentation with MLflow

### 4.5.1 Hyperparameter sweep — different learning rates

Three runs with LRs from `configs/lab04.yaml`, 5 epochs each. Each run uses a hierarchical name.

In [9]:
sweep_cfg = lab04_cfg['sweep']
sweep_results = []

# Rebuild loaders with sweep batch size
loader_cfg_sweep_train = {
    'batch_size': sweep_cfg['batch_size'],
    'num_workers': base_cfg['training']['num_workers'],
    'transform': train_transform,
}
loader_cfg_sweep_eval = {
    'batch_size': sweep_cfg['batch_size'],
    'num_workers': base_cfg['training']['num_workers'],
    'transform': eval_transform,
}
sweep_train_loader = create_data_loader(IMAGES_DIR, train_df, loader_cfg_sweep_train)
sweep_val_loader   = create_data_loader(IMAGES_DIR, val_df,   loader_cfg_sweep_eval)
sweep_test_loader  = create_data_loader(IMAGES_DIR, test_df,  loader_cfg_sweep_eval)

for i, lr in enumerate(sweep_cfg['learning_rates'], start=1):
    run_name = f'Experiment 2 - LR={lr}'
    save_path = Path(f'outputs/lab04_sweep_lr{str(lr).replace(".", "")}.pth')
    print(f'\n--- Starting run: {run_name} ---')

    sweep_model = CifarCNN(n_classes=base_cfg['data']['n_classes'])
    sweep_optimizer = torch.optim.Adam(sweep_model.parameters(), lr=lr)
    sweep_loss_fn = nn.CrossEntropyLoss()

    with mlflow.start_run(run_name=run_name) as run:
        sweep_run_id = run.info.run_id
        mlflow.log_params({
            'lr': lr,
            'batch_size': sweep_cfg['batch_size'],
            'num_epochs': sweep_cfg['num_epochs'],
            'optimizer': 'Adam',
            'n_classes': base_cfg['data']['n_classes'],
            'model': 'CifarCNN',
        })

        train_with_mlflow(
            model=sweep_model,
            train_loader=sweep_train_loader,
            val_loader=sweep_val_loader,
            loss_function=sweep_loss_fn,
            optimizer=sweep_optimizer,
            num_epochs=sweep_cfg['num_epochs'],
            device=device,
            save_path=save_path,
        )

        sweep_metrics = test_model(sweep_model, sweep_test_loader, sweep_loss_fn, device)
        mlflow.log_metrics({
            'test_loss':     sweep_metrics['loss'],
            'test_accuracy': sweep_metrics['accuracy'],
            'test_f1':       sweep_metrics['f1'],
        })
        mlflow.log_artifact(str(save_path), artifact_path='model')

        sweep_results.append({
            'run_name': run_name,
            'lr': lr,
            'accuracy': sweep_metrics['accuracy'],
            'f1': sweep_metrics['f1'],
            'test_loss': sweep_metrics['loss'],
        })
        print(f'  Accuracy: {sweep_metrics["accuracy"]:.4f} | F1: {sweep_metrics["f1"]:.4f}')


--- Starting run: Experiment 2 - LR=0.01 ---


2026-05-14 12:38:30,992 INFO Epoch 1/5, Train Loss: 3.1851


2026-05-14 12:38:39,574 INFO Epoch 1/5, Val Loss: 2.1621


2026-05-14 12:38:39,596 INFO Best model saved with val loss: 2.1621


2026-05-14 12:40:20,700 INFO Epoch 2/5, Train Loss: 1.9878


...

2026-05-14 13:05:02,112 INFO Epoch 5/5, Val Loss: 0.9965


2026-05-14 13:05:02,147 INFO Best model saved with val loss: 0.9965


2026-05-14 13:05:02,148 INFO Training complete.


2026-05-14 13:05:13,525 INFO Test Results — Loss: 1.0002 | Accuracy: 0.6497 | Precision: 0.6497 | Recall: 0.6497 | F1: 0.6416


  Accuracy: 0.6497 | F1: 0.6416


### 4.5.2 Compare runs

In [10]:
import pandas as pd

# Pull all runs from MLflow for programmatic comparison
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=['metrics.test_accuracy DESC'],
)

comparison_cols = ['tags.mlflow.runName', 'params.lr', 'params.num_epochs',
                   'metrics.test_accuracy', 'metrics.test_f1', 'metrics.test_loss']
available = [c for c in comparison_cols if c in runs_df.columns]
print('\n=== All Runs Comparison (sorted by accuracy) ===')
print(runs_df[available].to_string(index=False))
print(f'\nView in MLflow UI: mlflow server --backend-store-uri {TRACKING_URI} --host 0.0.0.0 --port 5000')


=== All Runs Comparison (sorted by accuracy) ===
                   tags.mlflow.runName params.lr params.num_epochs  metrics.test_accuracy  metrics.test_f1  metrics.test_loss
Experiment 1 - Stage 2: Model Training     0.001                10               0.711083         0.704910           0.820516
               Experiment 2 - LR=0.001     0.001                 5               0.656500         0.652161           0.973950
              Experiment 2 - LR=0.0001    0.0001                 5               0.649667         0.641577           1.000250
                Experiment 2 - LR=0.01      0.01                 5               0.387333         0.355100           1.562461

View in MLflow UI: mlflow server --backend-store-uri outputs/mlruns --host 0.0.0.0 --port 5000


## 4.6 Part 5: Lab Report

See [docs/lab_report_04.md](../docs/lab_report_04.md) for the full written report.